# AI DRP — TB Detection, Drug Resistance & Treatment (Prototype)

Cleaned, parameterized version of `ai_drp.py`.

> ⚠️ Research/educational prototype — **NOT** a medical device. Do not use for diagnosis or treatment decisions.


## 0. Configuration


In [ ]:
# Tunable parameters — change these instead of editing code below.
DATASET_SLUG = "tawsifurrahman/tuberculosis-tb-chest-xray-dataset"  # Kaggle dataset
DATASET_DIR = "TB_Chest_Radiography_Database"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_CNN = 10          # from-scratch CNN epochs (was 5)
EPOCHS_TRANSFER = 10     # MobileNetV2 transfer-learning epochs (was 5)
VALIDATION_SPLIT = 0.2
THRESHOLD = 0.5
CNN_MODEL_PATH = "tb_detection_model.h5"
TRANSFER_MODEL_PATH = "tb_detector_ai.h5"


## 1. Kaggle setup & dataset download
Upload your `kaggle.json` (Kaggle API token) when prompted.


In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset
!unzip -q tuberculosis-tb-chest-xray-dataset.zip


In [ ]:
import os
for folder in os.listdir(DATASET_DIR):
    p = os.path.join(DATASET_DIR, folder)
    if os.path.isdir(p):
        print(folder, len(os.listdir(p)))


## 2. Preprocessing & augmentation
Added light augmentation (flip/rotation) on the training split for better generalization.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT,
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=VALIDATION_SPLIT)

train_data = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="training")
val_data = val_datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="validation")


## 3a. Model A — from-scratch CNN


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

cnn = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(*IMG_SIZE, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
cnn.summary()
cnn.fit(train_data, validation_data=val_data, epochs=EPOCHS_CNN)
cnn.save(CNN_MODEL_PATH)


## 3b. Model B — MobileNetV2 transfer learning (recommended)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False
x = base.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
preds = Dense(1, activation="sigmoid")(x)
transfer = Model(inputs=base.input, outputs=preds)
transfer.compile(optimizer="adam", loss="binary_crossentropy",
                 metrics=["accuracy"])
transfer.summary()
transfer.fit(train_data, validation_data=val_data, epochs=EPOCHS_TRANSFER)
transfer.save(TRANSFER_MODEL_PATH)


## 4. Inference + Gradio demo
Loads the trained CNN and exposes a 4-output UI. Run this locally too once `tb_detection_model.h5` is in the repo root.


In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

model = tf.keras.models.load_model(CNN_MODEL_PATH)

def preprocess(img):
    img = img.convert("RGB").resize(IMG_SIZE)
    return np.expand_dims(np.array(img)/255.0, axis=0)

def predict_tb(image):
    pred = float(model.predict(preprocess(image), verbose=0)[0][0])
    if pred > THRESHOLD:
        return ("TB Detected", "rpoB mutation detected",
                "Rifampicin Resistant (Possible MDR-TB)",
                "Bedaquiline + Linezolid + Levofloxacin")
    return ("Normal", "No mutation detected", "Drug Sensitive",
            "Standard TB therapy")

gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=[gr.Textbox(label="TB Detection"),
             gr.Textbox(label="Mutation Analysis"),
             gr.Textbox(label="Drug Resistance Prediction"),
             gr.Textbox(label="Treatment Recommendation")],
    title="AI TB Detection + Drug Resistance + Treatment System",
).launch()
